# Run iQSM+ QSM reconstruction (Open Recon `qsm.py` module)

This mirrors [RunClientServerRecon.ipynb](RunClientServerRecon.ipynb), but wired specifically for
the `qsm.py` module: converts the example multi-echo GRE DICOMs (expected at
`data/dicoms`, gitignored -- populate it yourself) into MRD format, sends them to a
running server for QSM reconstruction via iQSM+ (https://github.com/sunhongfu/iQSM_Plus, cloned
locally into `iQSM_Plus/`, see [readme.md](readme.md)), and displays the result.

**Before running this notebook:** start the server using the **"Start QSM server"** launch
configuration (Run and Debug tab, top left) -- this runs `main.py -p 9020 -v -d qsm`, which
listens on port 9020 with `qsm` as the default config module.

## 1. Convert the example DICOMs to MRD format

In [ ]:
# Convert the example multi-echo GRE DICOMs (e.g. magnitude + phase, 160 slices x 5 echoes)
# into an MRD file. Expected at data/dicoms (gitignored, populate it yourself) --
# a repo-relative path, so this works the same in the devcontainer (workspace bind-mount
# covers the whole repo, including data/) and when running natively.
import os
import dicom2mrd
from types import SimpleNamespace

# Adjust the logging configuration otherwise INFO level isn't shown in notebook output
import logging
logging.getLogger().setLevel(logging.INFO)

dicomFolder = 'data/dicoms'
inputFilename = 'data/input_data.mrd'

dir, file = os.path.split(inputFilename)
if dir and not os.path.exists(dir):
    os.makedirs(dir)

if os.path.exists(inputFilename):
    overwrite = input(f'File {inputFilename} already exists!  Overwrite? y/[n]')
    if overwrite.lower() == 'y':
        os.remove(inputFilename)
    else:
        raise Exception('Aborting to not overwrite existing file')

args = SimpleNamespace(**dicom2mrd.defaults)
args.folder  = dicomFolder
args.outFile = inputFilename

dicom2mrd.main(args)

## 2. Run the QSM reconstruction

Make sure a QSM server is running before running this cell (this cell's `args.port` doesn't need
to change either way): open this repo in the devcontainer and use the **"Start QSM server"**
launch config (Run and Debug tab) -- `bet2` and GPU both work directly, on every host (see
[readme.md](readme.md)'s "Local testing" section for why). There's also a **"Start QSM server
(Docker)"** task (Terminal > Run Task) that runs the already-built `openrecon-qsm:prod` image
directly, if you'd rather not switch VS Code's environment.

Reconstruction takes several minutes (deep-learning inference over the full 3D multi-echo volume,
longer under the devcontainer's forced `--platform linux/amd64` emulation on Apple Silicon) --
this cell will block until it's done.

In [ ]:
# Send the converted data to the running QSM server
import client
from types import SimpleNamespace
import datetime
import logging
logging.basicConfig(format='%(asctime)s - %(message)s', level=logging.INFO)

outputFilename = 'data/qsm_recon.mrd'

args = SimpleNamespace(**client.defaults)
args.config    = 'qsm'
args.port      = 9020
args.out_group = str(datetime.datetime.now())
args.outfile   = outputFilename
args.filename  = inputFilename

client.main(args)


## 3. Display the QSM map

In [ ]:
# Display the reconstructed QSM map (and the source magnitude for comparison)
import h5py
import ismrmrd
import numpy as np
import matplotlib.pyplot as plt

with h5py.File(outputFilename, 'r') as d:
    group = list(d.keys())[-1]  # most recent reconstruction run

with ismrmrd.Dataset(outputFilename, group, False) as dset:
    subgroups = dset.list()
    imgGroups = [g for g in subgroups if g.startswith('image_') or g.startswith('images_')]
    print('Group %s contains %d image series:' % (group, len(imgGroups)))
    for g in imgGroups:
        print(f'  {g} ({dset.number_of_images(g)} images)')

    # qsm.py writes the QSM map at image_series_index=100 ("image_100"), alongside the
    # original buffered magnitude/phase images it now also passes through unmodified (see
    # the "keep original DICOM series" change) as image_0/image_1/image_2/etc. Must select
    # by name, not by position: imgGroups[-1] used to work when image_100 was the only
    # series, but "image_2" now sorts *after* "image_100" as a string ('2' > '1'), so
    # picking "last" silently grabs a passthrough series instead of the QSM map.
    qsmGroup = 'image_100' if 'image_100' in imgGroups else imgGroups[-1]
    n = dset.number_of_images(qsmGroup)
    midImg = dset.read_image(qsmGroup, n // 2)
    meta = ismrmrd.Meta.deserialize(midImg.attribute_string)

# qsm.py stores the QSM map as quantized uint16 (see its RescaleSlope/RescaleIntercept
# comment) -- apply the Modality LUT transform (real = raw * slope + intercept) to recover
# real ppm values before plotting, the same conversion any DICOM viewer applies
# automatically. Without this, imshow's vmin/vmax (in ppm) clip every raw uint16 pixel
# to the same end of the colormap, rendering as a blank image.
slope     = float(meta['RescaleSlope'])
intercept = float(meta['RescaleIntercept'])
qsmPpm    = np.squeeze(midImg.data).astype(np.float64) * slope + intercept

fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(qsmPpm, cmap='gray', vmin=-0.3, vmax=0.3)
ax.set_title(f'QSM (ppm), slice {n // 2 + 1}/{n}')
ax.axis('off')
plt.show()

print('MetaAttributes:')
for key, value in meta.items():
    print(f'  {key}: {value}')


## 4. Convert QSM results to DICOMs

In [ ]:
import mrd2dicom
from types import SimpleNamespace

args = SimpleNamespace(
    filename   = outputFilename,   # reuses the variable from your reconstruction cell
    in_group   = '',               # '' -> auto-select most recent run
    out_folder = 'data/qsm_recon_dicom',
)
mrd2dicom.main(args)